In [1]:
from langchain_groq import ChatGroq

In [2]:
llm = ChatGroq(
    temperature=0,
    model="llama-3.3-70b-versatile",
    groq_api_key='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'
)

response = llm.invoke("The first person on the moon was")
# print(response.content)
print(response.content)

The first person to set foot on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to walk on the moon.


In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/software-engineer/job/R-78383")
page_data = loader.load().pop().page_content
type(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.


str

In [4]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm     # Formatted prompt(rendered prompt) is sent to llm by creating a pipe using " | " operator
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

```json
{
  "role": "Software Engineer",
  "experience": "2 years",
  "skills": [
    "Consumer Order Management",
    "Databases (Oracle, AWS, RDS, AWS DynamoDB, AWS OpenSearch)",
    "Continuous Integration and Deployment (Jenkins, Cucumber)",
    "Infrastructure as Code (AWS CloudFormation, Terraform)",
    "Coding languages & technologies (Java, Spring Boot, Python)",
    "Observability (Splunk, Grafana, SignalFx, AWS CloudWatch)",
    "Container technologies (Docker, AWS ECS, AWS EKS)",
    "Cloud Networking and Security (Route 53, AWS ALB, Security Groups, WAF)",
    "Cloud Storage and governance (AWS S3, Glacier)",
    "Cloud Messaging (AWS SNS, AWS SQS, Apache Kafka)"
  ],
  "description": "Develop RESTful APIs, microservices, and cloud-based distributed systems aligned with Global Technology standards independently, with minimal supervision to meet defined digital product specifications. Advise Product Owners on discrete technology-related business problems, formulate options,

In [5]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Software Engineer',
 'experience': '2 years',
 'skills': ['Consumer Order Management',
  'Databases (Oracle, AWS, RDS, AWS DynamoDB, AWS OpenSearch)',
  'Continuous Integration and Deployment (Jenkins, Cucumber)',
  'Infrastructure as Code (AWS CloudFormation, Terraform)',
  'Coding languages & technologies (Java, Spring Boot, Python)',
  'Observability (Splunk, Grafana, SignalFx, AWS CloudWatch)',
  'Container technologies (Docker, AWS ECS, AWS EKS)',
  'Cloud Networking and Security (Route 53, AWS ALB, Security Groups, WAF)',
  'Cloud Storage and governance (AWS S3, Glacier)',
  'Cloud Messaging (AWS SNS, AWS SQS, Apache Kafka)'],
 'description': 'Develop RESTful APIs, microservices, and cloud-based distributed systems aligned with Global Technology standards independently, with minimal supervision to meet defined digital product specifications. Advise Product Owners on discrete technology-related business problems, formulate options, including assessing their relative meri

In [6]:
type(json_res)

dict

In [7]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [8]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])